# Diabetes Progression Predictor

Predicting one-year disease progression in diabetes patients from 10 baseline health measurements.

**Dataset:** scikit-learn's built-in `load_diabetes` (Efron et al., 2004).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

RANDOM_STATE = 42
%matplotlib inline

## 1. Load and inspect the data

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame
print(df.shape)
df.head()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
correlations = df.corr()["target"].drop("target").sort_values(ascending=False)
correlations

In [ ]:
plt.figure(figsize=(8, 5))
correlations.plot(kind="barh", color="#1F3864")
plt.title("Feature Correlation with Disease Progression")
plt.xlabel("Correlation coefficient")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["target"], bins=30, color="#1F3864", edgecolor="white")
plt.title("Distribution of Target (Disease Progression Score)")
plt.xlabel("Progression score")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.show()

## 3. Train/test split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## 4. Model 1 — Linear Regression (baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_r2 = r2_score(y_test, lr_preds)
print(f"RMSE: {lr_rmse:.2f} | MAE: {lr_mae:.2f} | R2: {lr_r2:.3f}")

## 5. Model 2 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)
print(f"RMSE: {rf_rmse:.2f} | MAE: {rf_mae:.2f} | R2: {rf_r2:.3f}")

## 6. Compare models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [lr_rmse, rf_rmse],
    "MAE": [lr_mae, rf_mae],
    "R2 Score": [lr_r2, rf_r2],
})
results

In [ ]:
best_preds = rf_preds if rf_r2 > lr_r2 else lr_preds
best_name = "Random Forest" if rf_r2 > lr_r2 else "Linear Regression"

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_preds, alpha=0.6, color="#1F3864")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Actual Progression Score")
plt.ylabel("Predicted Progression Score")
plt.title(f"Predicted vs Actual — {best_name}")
plt.tight_layout()
plt.show()

## 7. Feature importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color="#1F3864")
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Conclusion

Linear Regression slightly outperformed Random Forest on this dataset (R² of 0.453 vs 0.440).
With only 442 samples, there isn't enough data for the Random Forest to learn complex patterns
without overfitting — a good reminder that simpler models can win on small datasets.

`bmi` and `s5` had the strongest correlation with disease progression, consistent with known
clinical risk factors for diabetes.

**Next steps:** try Ridge/Lasso regularization, use k-fold cross-validation, and engineer
interaction features (e.g., bmi × bp).